# Chapter 15 — Problem Set 2: Solutions

Reference solutions for the advanced exercises in `problem_set_2.ipynb`. All solutions run offline.

---
*Generated by Berta AI*

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'scripts'))

import json
import time
import uuid
from pathlib import Path

import numpy as np
import pandas as pd

from monitoring import psi, ks_stat, Logger
from registry import ModelRegistry

print('imports OK')

## 1. Drift via PSI

In [ ]:
ref = pd.read_csv('../../datasets/reference_data.csv')
cur = pd.read_csv('../../datasets/current_data.csv')

def classify(score: float) -> str:
    if score < 0.10:
        return 'ok'
    if score < 0.25:
        return 'warning'
    return 'alert'

for col in ['feature_a', 'feature_b']:
    s = psi(ref[col].values, cur[col].values)
    print(f'  {col:<10}  PSI={s:.4f}  -> {classify(s)}')

## 2. Canary Splitter

In [ ]:
class CanarySplitter:
    def __init__(self, share: float, seed: int = 0):
        assert 0 <= share <= 1
        self.share = share
        self.seed = seed
    def route(self, request_id: str) -> str:
        h = abs(hash((self.seed, request_id))) % 10_000
        return 'candidate' if (h / 10_000) < self.share else 'production'

splitter = CanarySplitter(share=0.10, seed=42)

# Determinism: same id -> same routing
ids = [f'req-{i}' for i in range(10_000)]
routes = [splitter.route(i) for i in ids]
routes2 = [splitter.route(i) for i in ids]
assert routes == routes2

share = sum(1 for r in routes if r == 'candidate') / len(routes)
print(f'configured share: 0.10')
print(f'empirical  share: {share:.4f}')
assert abs(share - 0.10) < 0.01

## 3. CI Workflow with Eval Gates

In [ ]:
WORKFLOW = '''
name: ml-ci

on:
  push:
    branches: [ main ]
  pull_request:

jobs:
  ci:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip
      - run: pip install -r requirements.txt
      - name: Lint
        run: |
          pip install ruff
          ruff check scripts
      - name: Unit tests
        run: pytest tests/ -q
      - name: Smoke train
        run: python scripts/train.py --smoke
      - name: Eval gates
        run: |
          python -c "import json,sys; m=json.load(open(\"results/metrics.json\")); \\
            sys.exit(0 if (m[\"accuracy\"]>=0.80 and m[\"f1\"]>=0.75) else 1)"
      - name: Register
        if: github.ref == 'refs/heads/main'
        run: python scripts/register.py --stage Staging
'''.strip()
print(WORKFLOW)

## 4. Tiny Registry

In [ ]:
# Use a fresh subdir so the demo is hermetic.
import tempfile, shutil
tmp = Path(tempfile.mkdtemp())
artifact = tmp / 'fake.joblib'
artifact.write_bytes(b'pretend-this-is-a-pickle')

reg = ModelRegistry(tmp / 'registry')
reg.register('demo', '1.0.0', artifact, metrics={'f1': 0.81})

artifact2 = tmp / 'fake2.joblib'
artifact2.write_bytes(b'pretend-this-is-a-pickle-v2')
reg.register('demo', '2.0.0', artifact2, metrics={'f1': 0.85})

reg.transition_stage('demo', '1.0.0', 'Production')
print('after promoting v1 -> Production:', [(e.version, e.stage) for e in reg.list_models('demo')])

reg.transition_stage('demo', '2.0.0', 'Production')
print('after promoting v2 -> Production:', [(e.version, e.stage) for e in reg.list_models('demo')])

prod = reg.get_production('demo')
assert prod.version == '2.0.0'
v1 = reg.get('demo', '1.0.0')
assert v1.stage == 'Archived'
print('\nv1 was auto-archived; v2 is current Production.')

shutil.rmtree(tmp)

## 5. Structured Logging Middleware

In [ ]:
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient

log_path = Path('../../logs/req_log.jsonl')
if log_path.exists():
    log_path.unlink()
logger = Logger(log_path)

app = FastAPI()

@app.middleware('http')
async def log_requests(request: Request, call_next):
    rid = uuid.uuid4().hex[:12]
    t0 = time.perf_counter()
    response = await call_next(request)
    latency_ms = (time.perf_counter() - t0) * 1000
    logger.log(
        'http',
        request_id=rid,
        path=request.url.path,
        status_code=response.status_code,
        latency_ms=round(latency_ms, 3),
    )
    response.headers['x-request-id'] = rid
    return response

@app.post('/predict')
def predict(payload: dict):
    return {'prediction': 1, 'echo': payload}

client = TestClient(app)
for i in range(5):
    client.post('/predict', json={'feature_a': i, 'feature_b': i * 2})

records = logger.read_all()
print(f'logged {len(records)} requests')
for r in records:
    print('  ', r)

## 6. Rollback Policy

A worked answer (would be a markdown cell in the problem set):

### Triggers
| Signal | Threshold | Window |
|--------|-----------|--------|
| `error_rate` | > 1.5x baseline | 5 minutes |
| `p95_latency_ms` | > 1.5x baseline (or > 200 ms absolute) | 10 minutes |
| Eval regression on shadow data | F1 drop > 5% | rolling 1 hour |
| Drift PSI on key feature | > 0.25 | 1 hour |

Any **two** signals firing simultaneously, or any **one** at "critical" severity, triggers an automatic rollback.

### Action
1. Registry transition: candidate `Production -> Archived`, previous Production version reinstated.
2. Traffic-shift back to 100% on the prior version through the load balancer.
3. Disable any in-flight canary ramp.

### Paging
- **Severity high+**: page on-call ML engineer + service owner immediately.
- Runbook entry: `runbooks/model-rollback.md` with one-command rollback (e.g. `./scripts/rollback.sh demo-classifier`).

### Soak time
After rollback, wait **30 minutes** of nominal metrics before declaring the incident resolved.

### Post-mortem
- Timeline (alert → triage → rollback → resolved)
- Root cause (data, code, deps, infra)
- Detection gap (could we have caught it earlier?)
- Action items with owners and deadlines
- Update CI / monitoring to prevent recurrence